# Bronze Layer: Ingestion & Enrichment

Phase 1 pipeline — ingests raw CSVs into Bronze parquet, then loads the pre-scraped TMDB enrichment JSON into parquet.

### 1. Start a SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Bronze") \
    .master("local[*]") \
    .getOrCreate()

### 2. Define schemas

In [ ]:
from pyspark.sql.types import *

ratings_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("rating",       FloatType(),    True),
    StructField("timestamp",    LongType(),     True),
])

movies_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("title",    StringType(),   True),
    StructField("genres",   StringType(),   True),
])

links_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("imdbId",   IntegerType(),  True),
    StructField("tmdbId",   IntegerType(),  True),
])

tags_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("tag",          StringType(),   True),
    StructField("timestamp",    LongType(),     True),
])

enrichment_schema = StructType([
    StructField("movieId",           IntegerType()),
    StructField("tmdbId",            IntegerType()),
    StructField("title",             StringType()),
    StructField("directors",         ArrayType(StringType())),
    StructField("budget",            LongType()),
    StructField("revenue",           LongType()),
    StructField("runtime",           IntegerType()),
    StructField("release_date",      StringType()),
    StructField("poster_url",        StringType()),
    StructField("overview",          StringType()),
    StructField("vote_average",      FloatType()),
    StructField("original_language", StringType()),
])

### 3. Read each CSV

In [ ]:
df_ratings = spark.read.csv("ml-32m/ratings.csv", header=True, schema=ratings_schema)
df_movies  = spark.read.csv("ml-32m/movies.csv",  header=True, schema=movies_schema)
df_links   = spark.read.csv("ml-32m/links.csv",   header=True, schema=links_schema)
df_tags    = spark.read.csv("ml-32m/tags.csv",    header=True, schema=tags_schema)

### 4. Add ingestion metadata

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

def add_metadata(df, source_name):
    return df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_file", lit(source_name))

### 5. Write to bronze/ as parquet

In [ ]:
add_metadata(df_ratings, "ratings.csv") \
    .write.mode("overwrite").parquet("bronze/ratings")

add_metadata(df_movies, "movies.csv") \
    .write.mode("overwrite").parquet("bronze/movies")

add_metadata(df_links, "links.csv") \
    .write.mode("overwrite").parquet("bronze/links")

add_metadata(df_tags, "tags.csv") \
    .write.mode("overwrite").parquet("bronze/tags")

### 6. Sanity checks

In [ ]:
for name, df in [("ratings", df_ratings), ("movies", df_movies),
                  ("links", df_links), ("tags", df_tags)]:
    print(f"\n=== {name} ===")
    print(f"Rows: {df.count()}")
    df.printSchema()
    df.show(3)

---

## TMDB Enrichment

### 7. Load enrichment JSON to parquet

In [ ]:
import json

with open("bronze/enrichment/scraped_metadata.json") as f:
    records = json.load(f)

df_enrichment = (
    spark.createDataFrame(records, enrichment_schema)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("scraped_metadata.json"))
)

df_enrichment.write.mode("overwrite").parquet("bronze/enrichment/parquet")

print(f"Done. {df_enrichment.count()} rows written to bronze/enrichment/parquet")
df_enrichment.show(5, truncate=False)